[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/autogen/autogen_agents_prompt_chaining.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# AutoGen 提示词链智能体

本教程使用 [AutoGen 智能体](https://microsoft.github.io/autogen/0.2/docs/Getting-Started) 探索**提示词链**，并展示如何通过 Phoenix 追踪。
提示词链将复杂任务拆分为相互衔接的小任务，前一步输出成为下一步输入。它适用于能够明确拆分为固定子任务的场景，以增加延迟为代价，让每次 LLM 调用更简单、更准确，从而改善整体表现。

AutoGen 通过协调多个智能体，简化提示词链构建。每个 `AssistantAgent` 专注于专业任务，`UserProxyAgent` 管理对话流并传递步骤间的关键输出。Phoenix 追踪可以展示完整序列，监控各智能体调用，帮助调试链路。

本例创建多步骤市场分析工作流，从识别趋势到评估公司优势，全程自动执行并可追踪。

完成本教程后，你将学会：

- 设置专业 AutoGen 智能体

- 在智能体间传递上下文并串联任务

- 通过简化每次 LLM 调用提高任务准确性

- 使用 Phoenix 追踪并可视化完整执行过程

⚠️ 本教程需要 OpenAI 密钥。

## 设置密钥与依赖

In [ ]:
!pip install -qq pyautogen==0.9 autogen-agentchat~=0.2

In [ ]:
!pip install -qqq arize-phoenix arize-phoenix-otel openinference-instrumentation-openai

In [ ]:
import os
from getpass import getpass

import autogen

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

## 配置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="autogen-agents",
    auto_instrument=True,
)

## 提示词链示例任务：市场分析器

本例使用 AutoGen 智能体解决一系列串联的市场分析任务，并通过 Phoenix 捕获详细追踪。主用户智能体将各任务分配给专业智能体。

每个任务建立在前一步结果之上：智能体完成任务后，其输出摘要作为承接上下文传入下一个智能体的对话。这确保每个智能体都能获得此前收集的相关信息，让后续步骤利用前期研究。

## 定义智能体

首先定义四个专业 `AssistantAgents`，分别负责市场趋势分析、识别表现最佳的公司、公司比较，以及优劣势评估。再定义 `UserProxyAgent` 作为协调器，自动发送任务并管理整体对话流。`human_input_mode` 设为 "NEVER"，表示用户智能体自主运行，无需等待人工输入。

`llm_config` 指定全部助手智能体使用的配置。


![智能体设置](https://storage.googleapis.com/arize-phoenix-assets/assets/images/autogen_prompt_chaining.png)

In [ ]:
market_analysis_tasks = [
    """科技行业最新市场趋势是什么？重点使用新闻网站、研究博客和在线出版物等免费来源。""",
    """根据已识别的市场趋势，该领域目前领先的前 5 家公司是哪些？重点使用新闻网站、研究博客和在线出版物等免费来源。""",
    """比较这 5 家公司过去一年的股价表现、收入增长与产品创新。重点使用新闻网站、研究博客和在线出版物等免费来源。""",
    """总结这些公司在市场定位和未来增长潜力方面的优势与不足。""",
]

config_list = [
    {
        "model": "gpt-4o",
        "api_key": os.environ["OPENAI_API_KEY"],
    }
]

llm_config = {"config_list": config_list}

In [ ]:
market_trend_assistant = autogen.AssistantAgent(
    name="MarketTrendAssistant",
    llm_config=llm_config,
)

top_companies_assistant = autogen.AssistantAgent(
    name="TopCompaniesAssistant",
    llm_config=llm_config,
)

company_comparison_assistant = autogen.AssistantAgent(
    name="CompanyComparisonAssistant",
    llm_config=llm_config,
)

strength_weakness_assistant = autogen.AssistantAgent(
    name="StrengthWeaknessAssistant",
    llm_config=llm_config,
)

# 用户智能体
user = autogen.UserProxyAgent(
    name="User",
    human_input_mode="NEVER",
    is_termination_msg=lambda x: (
        x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE")
    ),
    code_execution_config={
        "last_n_messages": 1,
        "work_dir": "tasks",
        "use_docker": False,
    },
)

## 运行智能体
最后通过 `user.initiate_chats` 将每个任务发送给对应专业智能体。各智能体接收并处理具体任务，生成可传递给下一步的摘要。首个任务清除既有对话历史，后续任务使用 LLM 反思总结并传递关键上下文。

AutoGen 智能体除管理任务流外，还会生成并执行代码以完成目标。

In [ ]:
from opentelemetry.trace import StatusCode

tracer = tracer_provider.get_tracer(__name__)

# 启动各次对话
with tracer.start_as_current_span(
    "MarketAnalysisAgent",
    openinference_span_kind="agent",
) as agent_span:
    agent_span.set_status(StatusCode.OK)
    chat_results = user.initiate_chats(
        [
            {
                "recipient": market_trend_assistant,
                "message": market_analysis_tasks[0],
                "clear_history": True,
                "silent": False,
                "summary_method": "last_msg",
            },
            {
                "recipient": top_companies_assistant,
                "message": market_analysis_tasks[1],
                "summary_method": "reflection_with_llm",
            },
            {
                "recipient": company_comparison_assistant,
                "message": market_analysis_tasks[2],
                "summary_method": "reflection_with_llm",
            },
            {
                "recipient": strength_weakness_assistant,
                "message": market_analysis_tasks[3],
                "summary_method": "reflection_with_llm",
            },
        ]
    )

# 在 Phoenix 中查看结果

在 Phoenix 中可以跟踪提示词链的每次 LLM 调用如何利用先前输出，以及前序上下文如何指导后续回复。

追踪还展示 AutoGen 为获取和处理信息生成的代码，帮助理解系统如何与外部来源交互并完成任务，清楚呈现从推理到代码执行的过程，支持核对结果准确性。

你还可以查看延迟与令牌用量等元数据，寻找优化机会，提高智能体效率。


![结果动图](https://storage.googleapis.com/arize-phoenix-assets/assets/images/autogen_prompt_chaining.gif)

运行下方单元格查看完整追踪结果。

In [ ]:
from IPython.display import HTML

HTML("""
<video width="800" height="600" controls>
  <source src="https://storage.googleapis.com/arize-phoenix-assets/assets/images/autogen_prompt_chaining.mp4" type="video/mp4">
  浏览器不支持 video 标签。
</video>
""")